# Experiment: detect blank tiles by the point pip

A lettered tile prints its score in the top-right corner; a blank has no number.
A naive "dark pixels in the top-right" test **fails**, though: letters like `U`,
`H`, `R` have strokes that reach the same corner, so they look like they have a
pip.

The robust signal is **connected components**. The pip is a *small dark blob,
separate from the letter, sitting in the top-right and not touching any border*.
A letter's corner-reaching stroke is part of the one big letter component, so it
isn't a separate small blob. Border-touching blobs (edge strips, bleed from a
neighbouring tile) are ignored.

Verified on the real examples (`blank_E/U/H/R`) + all lettered tiles in
`frame.jpeg`: 15/15 correct.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from skimage.color import rgb2lab
from src.screenshot_to_map import WordfeudMap

WF = WordfeudMap('imgs/frame.jpeg')

def find_pip(tile, max_area=0.05, max_bot=0.45, min_left=0.62, L_thr=35):
    """Return the bbox (x, y, w, h) of the point pip, or None.

    The pip = a dark connected component that is small, lives in the top-right
    (bottom above ``max_bot``, left past ``min_left``), and does not touch any
    image border (which would mark it as an edge strip or neighbour bleed)."""
    img = tile[..., :3] if tile.shape[-1] == 4 else tile
    H, W = img.shape[:2]
    mask = (rgb2lab(img)[..., 0] < L_thr).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if x <= 0 or y <= 0 or x + w >= W or y + h >= H:      # border / bleed
            continue
        if area / (H * W) <= max_area and (y + h) / H <= max_bot and x / W >= min_left:
            return (x, y, w, h)
    return None

def is_blank(tile):
    return find_pip(tile) is None

## Lettered tiles — pip found (red box), verdict "letter"

In [ ]:
from matplotlib.patches import Rectangle

occ = [(r, c) for r in range(15) for c in range(15) if WF.map[r, c] == 1]
tiles = [(f'b{r},{c}', WF.tile_from_coord(r, c)) for r, c in occ] \
        + [(f'rack{i}', im) for i, im in enumerate(WF.available_letter_imgs())]

fig, axes = plt.subplots(1, len(tiles), figsize=(2.0 * len(tiles), 2.6))
for ax, (name, t) in zip(axes, tiles):
    pip = find_pip(t)
    ax.imshow(t); ax.axis('off')
    ax.set_title(f"{name}\n{'BLANK' if pip is None else 'letter'}")
    if pip:
        x, y, w, h = pip
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor='red', lw=2))
plt.tight_layout(); plt.show()

## Real blank examples (`blank_E/U/H/R`) — no pip, verdict "BLANK"
Note `H`, `U`, `R` have strokes in the top-right but no *separate* pip blob.

In [ ]:
names = ["E", "U", "H", "R"]
fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 3))
for ax, x in zip(axes, names):
    t = np.array(Image.open(f"imgs/blank_{x}.png").convert("RGB"))
    pip = find_pip(t)
    print(f"blank_{x}: {'BLANK' if pip is None else 'letter (!)'}")
    ax.imshow(t); ax.axis("off")
    ax.set_title(f"blank_{x}\n{'BLANK' if pip is None else 'letter (!)'}")
plt.tight_layout(); plt.show()

## Notes / integrating into the pipeline

- **Why components, not a dark-fraction:** letters whose strokes reach the
  top-right (`U`, `H`, `R`) defeat a pixel-count test. A connected component that
  is small + top-right + non-border is specifically the pip.
- Measured: lettered tiles have a pip blob (area ~0.01, `rows[0.04–0.35]`,
  `cols[0.73–0.89]`); blanks have none. Params (`max_area`, `max_bot`,
  `min_left`) have wide margins on the current samples.
- **To wire in** ([screenshot_to_map.py](src/screenshot_to_map.py)): in
  `map_with_letters`, when a tile is a letter, also call `is_blank(tile)` — keep
  the recognised glyph but flag it so scoring counts it as 0. In
  `available_letters`, emit `'*'` for a blank rack tile.
- Still only 4 blank samples — solid here, but add more (varied letters/boards)
  before fully trusting the params.